# Pilot 5 · Readable traces of rank-1 SFT and GRPO

Implemented: bootstrap and raw corpus preparation. Reuses saved adapters; no training or model capture here.

Use a fresh **CPU runtime** for this release. Run **1–2** to restore, **3** once for source auditing, then **7–9** for the corpus. Cells **4–6** are the completed environment/storage smoke path and need not be repeated. Do not use Run all.


## 1. Mount storage and load the pinned harness

CPU or GPU. The harness has a separate checkout; your saved experiment artifacts stay in Drive.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts')
EXPERIMENT_NAME = 'pilot5-readable-traces-v1'
HARNESS_COMMIT = 'd5f990fedc6bf614bfbbc011a02c0cdd3660f380'
REPO_DIR = Path('/content/pilot5-harness') / HARNESS_COMMIT

if not (REPO_DIR / '.git').is_dir():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', 'https://github.com/mahadikprasad15/spar-pilot.git',
                    str(REPO_DIR)], check=True)
    subprocess.run(['git', 'checkout', '--detach', HARNESS_COMMIT], cwd=REPO_DIR, check=True)
actual_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'],
                                        cwd=REPO_DIR, text=True).strip()
assert actual_commit == HARNESS_COMMIT, 'Harness checkout differs from its pin.'
loaded_package = sys.modules.get('pilot_eval')
if loaded_package and not Path(loaded_package.__file__).resolve().is_relative_to(REPO_DIR):
    raise RuntimeError('New harness release: restart the CPU runtime, then run 1–2 again.')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from pilot_eval.notebook_session import NotebookSession
runner = NotebookSession(ARTIFACT_ROOT, REPO_DIR, plan_name=EXPERIMENT_NAME)
print('Installing CPU audit/corpus libraries; model libraries are not required here.', flush=True)
runner.run([sys.executable, '-u', '-m', 'pip', 'install', '-e',
            str(REPO_DIR) + '[trace-audit,trace-corpus]'])


## 2. Restore saved state

Read-only. No model imports or earlier experiment stages run. “Audited config” records a previous source audit; capture will revalidate the source evidence.


In [ ]:
from pilot_eval.trace_session import TraceSession

session = TraceSession(ARTIFACT_ROOT, REPO_DIR, EXPERIMENT_NAME)
state = session.restore()
print('Smoke:', state['smoke']['state'])
print('Colab validation:', state['target_validation'])
for label, source in state['sources'].items():
    print(label, '|', source['state'], '|', source['path'])
print('Capture contract:', len(state['sites']), 'named sites; no capture launched.')
print('Raw corpus:', state['corpus']['state'])


## 3. Select and audit existing sources (CPU)

These are the existing prepared source configurations. Change a path only when selecting another reviewed source. Auditing checks their existing seals and source evidence; it does not load weights or generate answers.


In [ ]:
SOURCE_CONFIGS = {
    'sft': ARTIFACT_ROOT / 'plans/pilot3-inputs-v1/activation.prepared.json',
    'grpo': ARTIFACT_ROOT / 'plans/pilot4-grpo-fp32-v1-writes-v1/activation.prepared.json',
    'random': ARTIFACT_ROOT / 'plans/pilot4-grpo-fp32-v1-random-writes-v1/activation.prepared.json',
}
session.select_sources(SOURCE_CONFIGS)
source_audit = session.audit_sources()
for label, source in source_audit.items():
    print(label, '|', source['model'], '| checkpoints:', source['checkpoint_steps'])


## 4. Install and check pinned GPU dependencies

GPU smoke only. Removes conflicting optional vision/audio packages and imports the actual text-model classes in a fresh process. No model weights are downloaded. If Colab requests a restart, run 1–2 again before continuing.


In [ ]:
from pilot_eval.sft_backend import PINS

assert not session.active_children(), 'An experiment child is still running.'
runner.run([sys.executable, '-m', 'pip', 'install',
                *[f'{package}=={version}' for package, version in PINS.items()]], check=True)
runner.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                'torchvision', 'torchaudio'], check=True)
runtime = session.readiness()
print(json.dumps(runtime, indent=2))


## 5. Write the minimal Colab smoke artifact

Records actual import-check duration and GPU allocation, writes a small sealed artifact to Drive, then reads it back. After this cell finishes, restart the runtime. The GPU is not needed for cell 6.


In [ ]:
smoke = session.write_smoke()
print(json.dumps(smoke, indent=2))
print('Saved smoke:', session.directory)
print('Next: restart runtime, run 1–2, then 6.')


## 6. Verify storage after reconnect (CPU)

Run only in the fresh kernel after 1–2. Verifies the saved smoke seal and records a reconnect receipt. This qualifies bootstrap imports/storage only; capture and lens qualification are later tickets.


In [ ]:
restored = session.verify_reconnect()
print(json.dumps(restored['smoke'], indent=2))
print('Real-Colab bootstrap validation:', restored['target_validation'])


## 7. Review the raw corpus selection (CPU)

Requires cell 3’s saved source audit. Preserve the original 150 FineWeb identities; select 50 disjoint eligible documents from the same pinned 2,000-document pool. The seed and review are saved before sampling. “New-50” does not establish that documents were previously unseen.

Set `reviewed` to `True` only after approving the procedure and filling both notes. Legacy text/token discrepancies stop preparation; no replacement is automatic.


In [ ]:
state = session.restore()
assert all(state['sources'].get(label, {}).get('state', '').startswith('audited-config')
           for label in ['sft', 'grpo', 'random']), 'Run cell 3 to audit all three sources first.'

CORPUS_NAME = EXPERIMENT_NAME + '-raw-corpus-v1'
CORPUS_SOURCE = Path(state['sources']['sft']['path'])
CORPUS_REVIEW = {
    'schema_version': 1,
    'procedure': 'same-pool-disjoint-random-v1',
    'seed': 42,
    'reviewed': False,
    'notes': '',
    'prior_exposure': '',
}
# Notes describe the reviewed procedure. Exposure records known history or explicitly says unknown.
assert CORPUS_REVIEW['reviewed'] and CORPUS_REVIEW['notes'].strip() and CORPUS_REVIEW['prior_exposure'].strip(), (
    'Review the procedure and fill its notes/exposure history before preparation.'
)
CORPUS_REVIEW_PATH = ARTIFACT_ROOT / 'notebook' / EXPERIMENT_NAME / CORPUS_NAME / 'review.json'
from pilot_eval.workflow import _save_frozen
_save_frozen(CORPUS_REVIEW_PATH, CORPUS_REVIEW)
print('Reviewed corpus selection:', CORPUS_REVIEW_PATH)


## 8. Prepare or reuse the raw corpus (CPU)

Loads tokenizer files and streams the pinned FineWeb pool; no model weights or inference. Progress and errors are logged. A completed corpus is verified and reused without streaming or tokenizing again. An interrupted, unsealed preparation replays this bounded pool.


In [ ]:
runner.run([
    sys.executable, '-u', '-m', 'pilot_eval', 'traces-corpus-prepare',
    '--source-config', str(CORPUS_SOURCE), '--review', str(CORPUS_REVIEW_PATH),
    '--name', CORPUS_NAME, '--output-root', str(ARTIFACT_ROOT),
], env={'USE_TORCH': '0', 'USE_TF': '0'})
CORPUS_PATH = ARTIFACT_ROOT / 'plans' / CORPUS_NAME / 'traces.corpus.json'
session.select_corpus(CORPUS_PATH)
print('Frozen corpus:', CORPUS_PATH)


## 9. Verify saved corpus identities (CPU)

Checks corpus, tokenizer-file and input-parent hashes. Prints counts, tokenizer conventions and the new source indices. This verifies input preparation, not model capture or a scientific result. After a restart, run 1–2 and this cell to restore the selected corpus.


In [ ]:
state = session.restore()
assert state['corpus'].get('path'), 'Prepare and select the corpus in cell 8 first.'
CORPUS_PATH = Path(state['corpus']['path'])
runner.run([
    sys.executable, '-u', '-m', 'pilot_eval', 'traces-corpus-verify',
    '--config', str(CORPUS_PATH), '--output-root', str(ARTIFACT_ROOT),
])


## Next stages

Vocabulary/reveal-policy review, bounded capture qualification and community-lens qualification are subsequent tickets. Production capture and learned lens token lists are not implemented yet. The existing smoke artifacts remain valid for their recorded scope; they do not qualify numerical model capture.
